# TKAN Model Analysis and Improvement

This notebook addresses two key issues with the TKAN (Temporal Kernel Attention Network) model implementation:

1. **Inconsistent Results**: Each time the model is trained, it produces different results
2. **Improved Backtesting**: The current backtesting methodology can be enhanced

We'll analyze these issues and implement solutions.

## 1. Import Required Libraries

In [2]:
import os
import tensorflow as tf
from tkan import TKAN  # Ensure TKAN is correctly installed and compatible with TensorFlow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pandas.tseries.offsets import BDay
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit

# Set plotting style (using a valid matplotlib style)
plt.style.use('seaborn-v0_8-darkgrid')

# Define the path to your data file
data_lvc_path = r"c:\Personal\Business & Investments\Trading portfolio\Strategies\ETF TKAN\Data\LVC_daily.xlsx"


## 2. Set Random Seeds for Reproducibility

One of the primary reasons for inconsistent results between model runs is the lack of fixed random seeds. Setting random seeds ensures that the same "random" processes occur each time, making results reproducible.

In [3]:
import os

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)

print("Random seeds set for reproducibility")

# Define the path to your data file using relative path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
data_lvc_path = os.path.join(notebook_dir, "Data", "LVC_daily.xlsx")

Random seeds set for reproducibility


## 3. Data Processing

We'll use the same data processing function from your existing code, but with improved documentation.

In [4]:
def process_stock(data_path, train_test_split_date=None):
    """
    Process the stock data and prepare it for model training.
    
    Args:
        data_path: Path to the Excel file containing stock data
        train_test_split_date: Optional date to split training and testing data
        
    Returns:
        X: Feature DataFrame
        y: Target Series
        market_data: Complete market data DataFrame
    """
    # Load the data file
    market_data = pd.read_excel(data_path)
    
    # Display first few rows of the data to verify
    print(f"Loaded {len(market_data)} rows of data")
    print("First few rows:")
    display(market_data.head())
    
    # Check for missing values
    print(f"Missing values before processing:\n{market_data.isnull().sum()}")
    
    # Convert 'Date' to datetime and sort in ascending order
    market_data['Date'] = pd.to_datetime(market_data['Date'])
    market_data = market_data.sort_values('Date')  # Ensure data is sorted oldest first
    market_data.set_index('Date', inplace=True)
    
    # Create lagged features
    market_data['Prior Close Price'] = market_data['Close'].shift(1)
    market_data['Prior Volume'] = market_data['Volume'].shift(1)
    market_data['Prior High'] = market_data['High'].shift(1)
    market_data['Prior Low'] = market_data['Low'].shift(1)
    market_data['Prior SMAVG (15)'] = market_data['SMAVG (15)'].shift(1)
    
    # Add additional features to improve model performance
    # 1. Price change from previous day
    market_data['Price_Change'] = market_data['Close'] - market_data['Prior Close Price']
    # 2. Volatility measure (high-low spread)
    market_data['Volatility'] = market_data['High'] - market_data['Low']
    # 3. Price relative to moving average
    market_data['Price_MA_Ratio'] = market_data['Close'] / market_data['SMAVG (15)']
    
    # Drop rows with NaN values created by the shift
    market_data = market_data.dropna()
    
    print(f"Data after processing: {len(market_data)} rows")
    
    # Verify that the DataFrame is not empty
    if market_data.empty:
        raise ValueError(
            "After creating lagged features and dropping NaNs, the DataFrame is empty. Check your data.")

    # Separate features and target
    features = [
        'Prior Close Price', 'Prior Volume', 'Prior High', 'Prior Low', 
        'Prior SMAVG (15)', 'Price_Change', 'Volatility', 'Price_MA_Ratio'
    ]
    X = market_data[features]
    y = market_data['Close']
    
    # Split data if a split date is provided
    if train_test_split_date is not None:
        split_date = pd.to_datetime(train_test_split_date)
        train_data = market_data[market_data.index < split_date]
        test_data = market_data[market_data.index >= split_date]
        
        X_train = train_data[features]
        y_train = train_data['Close']
        X_test = test_data[features]
        y_test = test_data['Close']
        
        print(f"Train set: {len(X_train)} samples ({X_train.index.min()} to {X_train.index.max()})")
        print(f"Test set: {len(X_test)} samples ({X_test.index.min()} to {X_test.index.max()})")
        
        return X_train, y_train, X_test, y_test, market_data
        
    return X, y, market_data

## 4. Load and Process the Data

In [5]:
# Process data with train/test split
X_train, y_train, X_test, y_test, market_data = process_stock(
    data_lvc_path, 
    train_test_split_date='2020-07-01'  # Same as original backtest start date
)

Loaded 1631 rows of data
First few rows:


,Date,High,Low,Close,Volume,SMAVG (15)
0,2019-09-09,21.610,21.37,21.465,197525,21.465000
1,2019-09-10,21.500,21.19,21.455,117600,21.460000
2,2019-09-11,21.730,21.55,21.660,230081,21.526667
3,2019-09-12,22.055,21.49,21.850,222528,21.607500
4,2019-09-13,22.080,21.84,21.950,203636,21.676000


Missing values before processing:
Date          0
High          0
Low           0
Close         0
Volume        0
SMAVG (15)    0
dtype: int64
Data after processing: 1630 rows
Train set: 205 samples (2019-09-10 00:00:00 to 2020-06-30 00:00:00)
Test set: 1425 samples (2020-07-01 00:00:00 to 2026-01-26 00:00:00)


## 5. Create Sequences with Improved Function

In [6]:
def create_sequences(X, y, window_size, prediction_days=10):
    """
    Create sequences for time-series prediction with improved documentation.
    
    Args:
        X (ndarray): Scaled feature array
        y (ndarray): Scaled target array
        window_size (int): Number of past days to consider for each sequence
        prediction_days (int): Number of future days to predict
        
    Returns:
        X_seq (ndarray): Feature sequences
        y_seq (ndarray): Target sequences
    """
    X_seq, y_seq = [], []
    
    # Ensure there's enough data for at least one sequence
    if len(X) <= window_size:
        raise ValueError(f"Not enough data points ({len(X)}) for window_size={window_size}")
    
    # Create sequences
    for i in range(len(X) - window_size - prediction_days + 1):
        # Input sequence: window_size days of features
        X_seq.append(X[i:i + window_size])
        
        # Target sequence: prediction_days days of prices after the window
        target_indices = range(i + window_size, i + window_size + prediction_days)
        y_window = [y[j] for j in target_indices if j < len(y)]
        
        # Only add sequences where we have complete prediction_days target values
        if len(y_window) == prediction_days:
            y_seq.append(y_window)
    
    return np.array(X_seq), np.array(y_seq)

## 6. Normalize Data and Create Sequences

In [7]:
# Define parameters
window_size = 10
prediction_days = 10

# Initialize scalers
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit scalers on training data
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_array = y_train.values.reshape(-1, 1)
y_train_scaled = scaler_y.fit_transform(y_train_array).flatten()

# Scale test data using the same scalers
X_test_scaled = scaler_X.transform(X_test)
y_test_array = y_test.values.reshape(-1, 1)
y_test_scaled = scaler_y.transform(y_test_array).flatten()

# Create sequences
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled, y_train_scaled, window_size, prediction_days
)
X_test_seq, y_test_seq = create_sequences(
    X_test_scaled, y_test_scaled, window_size, prediction_days
)

print(f"Training sequences: {X_train_seq.shape}, {y_train_seq.shape}")
print(f"Testing sequences: {X_test_seq.shape}, {y_test_seq.shape}")

Training sequences: (186, 10, 8), (186, 10)
Testing sequences: (1406, 10, 8), (1406, 10)


## 7. Improved Model Architecture with Dropout for Regularization

In [8]:
def build_improved_model(input_shape, dropout_rate=0.2):
    """
    Build an improved TKAN model with dropout for regularization
    
    Args:
        input_shape: Shape of input data (window_size, n_features)
        dropout_rate: Dropout rate for regularization
        
    Returns:
        Compiled TensorFlow model
    """
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(shape=input_shape),
        
        # TKAN Layers with dropout for regularization
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(dropout_rate),
        
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(dropout_rate),
        
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(dropout_rate),
        
        tf.keras.layers.Dense(1)  # Output layer
    ])
    
    # Use Adam optimizer with a lower learning rate for stability
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )
    
    return model

## 8. Model Training with Early Stopping

In [9]:
# Input shape from training data
input_shape = X_train_seq.shape[1:]
print(f"Input shape: {input_shape}")

# Build the model
model = build_improved_model(input_shape, dropout_rate=0.2)
model.summary()

# Set up callbacks for early stopping and model checkpoints
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train the model with early stopping
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=100,  # Maximum epochs (will stop earlier with early stopping)
    batch_size=32,
    validation_split=0.1,  # Use 10% of training data for validation
    callbacks=[early_stopping],
    verbose=1
)

Input shape: (10, 8)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tkan (TKAN)                     │ (None, 10, 100)        │        33,706 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tkan_1 (TKAN)                   │ (None, 10, 100)        │        71,610 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tkan_2 (TKAN)                   │ (None, 10, 100)        │        71,610 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 1)          │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177,027 (691.51 KB)

 Trainable params: 176,997 (691.39 KB)

 Non-trainable params: 30 (120.00 B)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 16s 318ms/step - loss: 0.5806 - mae: 0.6405 - val_loss: 0.2826 - val_mae: 0.4701
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.3533 - mae: 0.4819 - val_loss: 0.2805 - val_mae: 0.4714
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.2489 - mae: 0.4064 - val_loss: 0.0393 - val_mae: 0.1740
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.2527 - mae: 0.4253 - val_loss: 0.0231 - val_mae: 0.1189
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.2358 - mae: 0.4136 - val_loss: 0.0682 - val_mae: 0.2380
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.2040 - mae: 0.3734 - val_loss: 0.1101 - val_mae: 0.3034
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.1965 - mae: 0.3642 - val_loss: 0.0615 - val_mae: 0.2285
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.1898 - mae: 0.3681 - val_loss: 0.0364 - val_mae: 0.1761
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.1859 - mae

## 9. Plot Training History

In [ ]:
def plot_training_history(history):
    """Plot the training and validation loss/metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    
    # Plot loss
    ax1.plot(history.history['loss'], label='Training Loss', color='blue')
    ax1.plot(history.history['val_loss'], label='Validation Loss', color='orange')
    ax1.set_title('Model Loss Over Epochs')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss (MSE)')
    ax1.legend()
    ax1.grid(True)
    
    # Plot MAE
    ax2.plot(history.history['mae'], label='Training MAE', color='blue')
    ax2.plot(history.history['val_mae'], label='Validation MAE', color='orange')
    ax2.set_title('Model MAE Over Epochs')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Mean Absolute Error')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot the training history
plot_training_history(history)

## 10. Evaluate Model Performance

In [ ]:
# Make predictions on test data
test_predictions = model.predict(X_test_seq)

# Reshape predictions to match the original shape for inverse transform
test_preds_reshaped = test_predictions.reshape(-1, 1)
y_test_seq_reshaped = y_test_seq.reshape(-1, 1)

# Inverse transform to get actual prices
test_preds_actual = scaler_y.inverse_transform(test_preds_reshaped)
y_test_actual = scaler_y.inverse_transform(y_test_seq_reshaped)

# Calculate error metrics
mae = mean_absolute_error(y_test_actual, test_preds_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, test_preds_actual))
mape = np.mean(np.abs((y_test_actual - test_preds_actual) / y_test_actual)) * 100

print(f"Mean Absolute Error (MAE): £{mae:.2f}")
print(f"Root Mean Squared Error (RMSE): £{rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

## 11. Prediction Confidence Analysis using Monte Carlo Dropout

One of the main issues with your current model is that it doesn't measure prediction confidence. Using Monte Carlo Dropout, we can get multiple predictions and measure the uncertainty.

In [ ]:
def monte_carlo_predictions(model, X_input, n_iterations=30):
    """
    Generate multiple predictions using Monte Carlo Dropout
    
    Args:
        model: Trained TensorFlow model with dropout layers
        X_input: Input data
        n_iterations: Number of prediction iterations
        
    Returns:
        predictions: List of predictions
    """
    predictions = []
    
    # Make multiple predictions with dropout enabled
    for _ in range(n_iterations):
        pred = model(X_input, training=True).numpy()
        predictions.append(pred)
        
    return predictions

# Select a sample from the test set
sample_idx = 10
sample_input = X_test_seq[sample_idx:sample_idx+1]

# Generate multiple predictions
mc_predictions = monte_carlo_predictions(model, sample_input, n_iterations=50)

# Convert to numpy array for easier manipulation
mc_predictions_array = np.array(mc_predictions).squeeze()

# Calculate statistics
mean_predictions = np.mean(mc_predictions_array, axis=0)
std_predictions = np.std(mc_predictions_array, axis=0)

# Convert to actual price scale
mean_actual = scaler_y.inverse_transform(mean_predictions.reshape(-1, 1)).flatten()
std_actual = std_predictions * (scaler_y.data_max_ - scaler_y.data_min_)
upper_bound = scaler_y.inverse_transform((mean_predictions + 2*std_predictions).reshape(-1, 1)).flatten()
lower_bound = scaler_y.inverse_transform((mean_predictions - 2*std_predictions).reshape(-1, 1)).flatten()

# Plot the predictions with confidence intervals
days = np.arange(1, prediction_days + 1)

plt.figure(figsize=(12, 6))
plt.plot(days, mean_actual, 'b-', label='Predicted Price')
plt.fill_between(days, lower_bound, upper_bound, alpha=0.2, color='blue', label='95% Confidence Interval')

# Add the actual price from test data for comparison
actual_price = scaler_y.inverse_transform(y_test_seq[sample_idx].reshape(-1, 1)).flatten()
plt.plot(days, actual_price, 'r--', label='Actual Price')

plt.title('Prediction with Confidence Intervals')
plt.xlabel('Days Ahead')
plt.ylabel('Price (£)')
plt.legend()
plt.grid(True)
plt.show()

# Create a table with the confidence metrics
confidence_df = pd.DataFrame({
    'Day': days,
    'Predicted Price': mean_actual,
    'Actual Price': actual_price,
    'Standard Deviation': std_actual.flatten(),
    'Lower Bound (95%)': lower_bound,
    'Upper Bound (95%)': upper_bound,
    'Confidence (%)': 100 * (1 - std_actual.flatten() / mean_actual)
})

display(confidence_df)

## 12. Improved Backtesting Strategy

Now we'll implement an improved backtesting strategy that includes:
1. Confidence-based trading decisions
2. Risk management (stop loss, take profit)
3. Position sizing based on confidence

In [ ]:
def improved_backtest(model, X, y, market_data, scaler_X, scaler_y, window_size=10, prediction_days=10, 
                      initial_capital=100000, confidence_threshold=0.7, max_position_size=0.3,
                      take_profit_pct=0.05, stop_loss_pct=0.03):
    """
    Perform backtesting with improved risk management and confidence assessment
    
    Args:
        model: Trained TensorFlow model
        X: Feature DataFrame
        y: Target Series
        market_data: Market data DataFrame
        scaler_X: Fitted feature scaler
        scaler_y: Fitted target scaler
        window_size: Number of past days to consider for prediction
        prediction_days: Number of days to predict ahead
        initial_capital: Starting capital
        confidence_threshold: Minimum confidence required to trade
        max_position_size: Maximum position size as percentage of capital
        take_profit_pct: Take profit level as percentage of entry price
        stop_loss_pct: Stop loss level as percentage of entry price
        
    Returns:
        trades: List of trade dictionaries
        portfolio_history: Daily portfolio value DataFrame
    """
    trades = []
    portfolio_history = []
    holding = False
    buy_price = 0
    buy_date = None
    shares = 0
    capital = initial_capital
    stop_loss_price = 0
    take_profit_price = 0
    trade_confidence = 0
    
    dates = X.index
    total_days = len(dates)
    
    for current_idx in range(window_size, total_days - prediction_days):
        current_date = dates[current_idx]
        current_price = y[current_idx]
        
        # Record portfolio value
        portfolio_value = capital + shares * current_price
        portfolio_history.append({
            'Date': current_date,
            'Portfolio Value': portfolio_value,
            'Cash': capital,
            'Stock Value': shares * current_price,
            'Shares': shares
        })
        
        if holding:
            # Check for stop loss or take profit
            if current_price <= stop_loss_price:
                # Stop loss triggered
                profit = (current_price - buy_price) * shares
                capital += shares * current_price
                
                trades[-1]['Sell Date'] = current_date
                trades[-1]['Sell Price'] = current_price
                trades[-1]['Profit'] = profit
                trades[-1]['Trade Type'] = 'Stop Loss'
                trades[-1]['Days Held'] = (current_date - buy_date).days
                
                holding = False
                shares = 0
                buy_price = 0
                buy_date = None
                stop_loss_price = 0
                take_profit_price = 0
                
            elif current_price >= take_profit_price:
                # Take profit triggered
                profit = (current_price - buy_price) * shares
                capital += shares * current_price
                
                trades[-1]['Sell Date'] = current_date
                trades[-1]['Sell Price'] = current_price
                trades[-1]['Profit'] = profit
                trades[-1]['Trade Type'] = 'Take Profit'
                trades[-1]['Days Held'] = (current_date - buy_date).days
                
                holding = False
                shares = 0
                buy_price = 0
                buy_date = None
                stop_loss_price = 0
                take_profit_price = 0
        else:
            # Check for new trading opportunities
            X_window = X.iloc[current_idx - window_size:current_idx].values
            X_window_scaled = scaler_X.transform(X_window).reshape(1, window_size, -1)
            
            # Use Monte Carlo Dropout for confident predictions
            mc_predictions = monte_carlo_predictions(model, X_window_scaled, n_iterations=30)
            mc_predictions_array = np.array(mc_predictions).squeeze()
            
            # Calculate statistics
            mean_predictions = np.mean(mc_predictions_array, axis=0)
            std_predictions = np.std(mc_predictions_array, axis=0)
            
            # Convert to actual price scale
            mean_actual = scaler_y.inverse_transform(mean_predictions.reshape(-1, 1)).flatten()
            std_actual = std_predictions * (scaler_y.data_max_ - scaler_y.data_min_).flatten()
            
            # Calculate confidence scores (lower std = higher confidence)
            confidence_scores = 1 - (std_actual / mean_actual)
            
            # Find best day to trade based on expected return and confidence
            returns = (mean_actual - current_price) / current_price
            risk_adjusted_returns = returns * confidence_scores
            
            # Only consider days with positive return
            positive_days = np.where(returns > 0.01)[0]  # At least 1% return
            
            if len(positive_days) > 0:
                # Find the best day (highest risk-adjusted return with sufficient confidence)
                confident_days = positive_days[confidence_scores[positive_days] > confidence_threshold]
                
                if len(confident_days) > 0:
                    best_day_idx = confident_days[np.argmax(risk_adjusted_returns[confident_days])]
                    best_day_ahead = best_day_idx + 1  # convert to 1-indexed day number
                    expected_price = mean_actual[best_day_idx]
                    predicted_date = current_date + pd.Timedelta(days=best_day_ahead)
                    
                    # Check if prediction date is within market_data
                    if current_idx + best_day_ahead < len(dates):
                        # Calculate position size based on confidence
                        position_confidence = confidence_scores[best_day_idx]
                        position_size = max(0.1, min(max_position_size, position_confidence))
                        
                        # Execute the trade
                        cash_to_invest = capital * position_size
                        shares = cash_to_invest // current_price  # Integer division
                        
                        if shares > 0:
                            # Record trade entry
                            buy_price = current_price
                            buy_date = current_date
                            capital -= shares * buy_price
                            trade_confidence = position_confidence
                            
                            # Set stop loss and take profit levels
                            stop_loss_price = buy_price * (1 - stop_loss_pct)
                            take_profit_price = buy_price * (1 + take_profit_pct)
                            
                            # Record trade
                            trades.append({
                                'Buy Date': buy_date,
                                'Buy Price': buy_price,
                                'Predicted Date': predicted_date,
                                'Predicted Price': expected_price,
                                'Confidence': position_confidence,
                                'Shares': shares,
                                'Position Size': position_size,
                                'Stop Loss': stop_loss_price,
                                'Take Profit': take_profit_price,
                                'Trade Type': 'Entry'
                            })
                            
                            holding = True
    
    # Close any open position at the end of the backtest
    if holding:
        final_date = dates[-1]
        final_price = y.iloc[-1]
        profit = (final_price - buy_price) * shares
        capital += shares * final_price
        
        trades[-1]['Sell Date'] = final_date
        trades[-1]['Sell Price'] = final_price
        trades[-1]['Profit'] = profit
        trades[-1]['Trade Type'] = 'Final Close'
        trades[-1]['Days Held'] = (final_date - buy_date).days
        
        # Final portfolio value
        portfolio_history.append({
            'Date': final_date,
            'Portfolio Value': capital,
            'Cash': capital,
            'Stock Value': 0,
            'Shares': 0
        })
    
    return trades, pd.DataFrame(portfolio_history)

## 13. Run Improved Backtest

In [ ]:
# Run the improved backtest
trades, portfolio_history = improved_backtest(
    model, X_test, y_test, market_data, scaler_X, scaler_y,
    window_size=window_size,
    prediction_days=prediction_days,
    initial_capital=100000,
    confidence_threshold=0.7,
    max_position_size=0.3,
    take_profit_pct=0.05,
    stop_loss_pct=0.03
)

# Display trade summary
if trades:
    trades_df = pd.DataFrame(trades)
    display(trades_df)
    
    # Calculate performance statistics
    winning_trades = trades_df[trades_df.get('Profit', 0) > 0]
    losing_trades = trades_df[trades_df.get('Profit', 0) <= 0]
    
    total_trades = len(trades_df)
    winning_ratio = len(winning_trades) / total_trades * 100 if total_trades > 0 else 0
    total_profit = trades_df['Profit'].sum() if 'Profit' in trades_df else 0
    roi = (total_profit / 100000) * 100
    
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {len(winning_trades)} ({winning_ratio:.2f}%)")
    print(f"Losing Trades: {len(losing_trades)} ({100-winning_ratio:.2f}%)")
    print(f"Total Profit: £{total_profit:.2f}")
    print(f"ROI: {roi:.2f}%")
    
    # Average holding period
    if 'Days Held' in trades_df:
        avg_holding_period = trades_df['Days Held'].mean()
        print(f"Average Holding Period: {avg_holding_period:.2f} days")
else:
    print("No trades were executed during the backtest period.")

## 14. Visualize Backtest Results

In [ ]:
def plot_backtest_results(trades_df, portfolio_history, market_data, initial_capital):
    """
    Visualize backtest results including trades, equity curve, and drawdowns
    
    Args:
        trades_df: DataFrame of executed trades
        portfolio_history: DataFrame with daily portfolio values
        market_data: Original market data
        initial_capital: Starting capital
    """
    # Create a figure with 3 subplots
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 16), gridspec_kw={'height_ratios': [2, 1, 1]})
    
    # 1. Price chart with buy/sell signals
    test_dates = portfolio_history['Date']
    test_prices = market_data.loc[test_dates, 'Close'] if all(d in market_data.index for d in test_dates) else []
    
    ax1.plot(portfolio_history['Date'], test_prices, label='Price', color='blue')
    
    if not trades_df.empty and 'Buy Date' in trades_df and 'Sell Date' in trades_df:
        # Plot buy signals
        buy_dates = trades_df['Buy Date']
        buy_prices = trades_df['Buy Price']
        ax1.scatter(buy_dates, buy_prices, marker='^', color='green', label='Buy', s=100)
        
        # Plot sell signals
        sell_dates = trades_df['Sell Date']
        sell_prices = trades_df['Sell Price']
        
        # Color the sell markers based on profit/loss
        for i, (date, price, profit) in enumerate(zip(sell_dates, sell_prices, trades_df['Profit'])):
            color = 'green' if profit > 0 else 'red'
            marker = 'o' if profit > 0 else 'v'
            ax1.scatter(date, price, marker=marker, color=color, s=100)
        
        ax1.scatter([], [], marker='o', color='green', label='Profit', s=100)
        ax1.scatter([], [], marker='v', color='red', label='Loss', s=100)
    
    ax1.set_title('Price Chart with Trading Signals')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Price (£)')
    ax1.legend()
    ax1.grid(True)
    
    # 2. Portfolio value over time
    ax2.plot(portfolio_history['Date'], portfolio_history['Portfolio Value'], 
             label='Portfolio Value', color='purple')
    ax2.axhline(y=initial_capital, color='gray', linestyle='--', label=f"Initial Capital (£{initial_capital:,.0f})")
    ax2.set_title('Portfolio Value Over Time')
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Value (£)')
    ax2.legend()
    ax2.grid(True)
    
    # 3. Drawdown analysis
    portfolio_history['Previous Peak'] = portfolio_history['Portfolio Value'].cummax()
    portfolio_history['Drawdown'] = (portfolio_history['Portfolio Value'] / portfolio_history['Previous Peak'] - 1) * 100
    
    ax3.fill_between(portfolio_history['Date'], portfolio_history['Drawdown'], 0, 
                     color='red', alpha=0.3, label='Drawdown')
    ax3.set_title('Portfolio Drawdown')
    ax3.set_xlabel('Date')
    ax3.set_ylabel('Drawdown (%)')
    ax3.set_ylim(min(portfolio_history['Drawdown']) - 1, 1)
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Return key metrics
    max_drawdown = portfolio_history['Drawdown'].min()
    final_value = portfolio_history['Portfolio Value'].iloc[-1]
    total_return = (final_value / initial_capital - 1) * 100
    
    print(f"Max Drawdown: {max_drawdown:.2f}%")
    print(f"Final Portfolio Value: £{final_value:,.2f}")
    print(f"Total Return: {total_return:.2f}%")
    
    # Calculate Sharpe Ratio (assuming risk-free rate of 0%)
    portfolio_history['Daily Returns'] = portfolio_history['Portfolio Value'].pct_change()
    avg_daily_return = portfolio_history['Daily Returns'].mean()
    daily_std = portfolio_history['Daily Returns'].std()
    sharpe_ratio = np.sqrt(252) * (avg_daily_return / daily_std) if daily_std > 0 else 0
    
    print(f"Annualized Sharpe Ratio: {sharpe_ratio:.2f}")

# Call the function to plot results
if trades:
    plot_backtest_results(pd.DataFrame(trades), portfolio_history, market_data, initial_capital=100000)

## 15. Compare Different Trading Strategies

Now let's compare multiple trading strategies with different parameters to find the optimal approach.

In [ ]:
# Define different strategy parameters
strategies = [
    {
        'name': 'Conservative',
        'confidence_threshold': 0.8,
        'max_position_size': 0.2,
        'take_profit_pct': 0.04,
        'stop_loss_pct': 0.02
    },
    {
        'name': 'Balanced',
        'confidence_threshold': 0.7,
        'max_position_size': 0.3,
        'take_profit_pct': 0.05,
        'stop_loss_pct': 0.03
    },
    {
        'name': 'Aggressive',
        'confidence_threshold': 0.6,
        'max_position_size': 0.4,
        'take_profit_pct': 0.07,
        'stop_loss_pct': 0.05
    }
]

# Test different strategies
results = []

for strategy in strategies:
    print(f"\nTesting {strategy['name']} strategy...")
    trades, portfolio_history = improved_backtest(
        model, X_test, y_test, market_data, scaler_X, scaler_y,
        window_size=window_size,
        prediction_days=prediction_days,
        initial_capital=100000,
        confidence_threshold=strategy['confidence_threshold'],
        max_position_size=strategy['max_position_size'],
        take_profit_pct=strategy['take_profit_pct'],
        stop_loss_pct=strategy['stop_loss_pct']
    )
    
    # Calculate metrics
    if trades:
        trades_df = pd.DataFrame(trades)
        total_trades = len(trades_df)
        winning_trades = len(trades_df[trades_df.get('Profit', 0) > 0])
        winning_ratio = winning_trades / total_trades * 100 if total_trades > 0 else 0
        total_profit = trades_df['Profit'].sum() if 'Profit' in trades_df else 0
        roi = (total_profit / 100000) * 100
        
        # Calculate max drawdown
        portfolio_history['Previous Peak'] = portfolio_history['Portfolio Value'].cummax()
        portfolio_history['Drawdown'] = (portfolio_history['Portfolio Value'] / portfolio_history['Previous Peak'] - 1) * 100
        max_drawdown = portfolio_history['Drawdown'].min()
        
        # Calculate Sharpe Ratio
        portfolio_history['Daily Returns'] = portfolio_history['Portfolio Value'].pct_change()
        avg_daily_return = portfolio_history['Daily Returns'].mean()
        daily_std = portfolio_history['Daily Returns'].std()
        sharpe_ratio = np.sqrt(252) * (avg_daily_return / daily_std) if daily_std > 0 else 0
        
        result = {
            'Strategy': strategy['name'],
            'Total Trades': total_trades,
            'Winning Trades': winning_trades,
            'Winning Ratio (%)': winning_ratio,
            'Total Profit (£)': total_profit,
            'ROI (%)': roi,
            'Max Drawdown (%)': max_drawdown,
            'Sharpe Ratio': sharpe_ratio
        }
        results.append(result)
        print(f"  Total Trades: {total_trades}")
        print(f"  Win Rate: {winning_ratio:.2f}%")
        print(f"  Total Profit: £{total_profit:.2f}")
        print(f"  ROI: {roi:.2f}%")
        print(f"  Max Drawdown: {max_drawdown:.2f}%")
        print(f"  Sharpe Ratio: {sharpe_ratio:.2f}")
    else:
        print("  No trades executed")

# Display comparison results
if results:
    results_df = pd.DataFrame(results)
    display(results_df)

## 16. Conclusion and Recommendations

Based on our analysis, we can provide several recommendations to improve your TKAN model and backtesting strategy:

1. **Model Consistency**:
   - Always set random seeds for reproducibility
   - Use dropout layers for regularization and uncertainty estimation
   - Implement early stopping to prevent overfitting

2. **Improved Backtesting**:
   - Use Monte Carlo dropout for prediction confidence
   - Implement risk management (stop-loss, take-profit)
   - Adjust position sizing based on confidence
   - Calculate and monitor key performance metrics

3. **Additional Enhancements**:
   - Use time-series cross-validation for more robust model evaluation
   - Add more features based on technical indicators
   - Implement a walk-forward backtesting approach for more realistic results

By implementing these improvements, your TKAN model should provide more consistent and reliable predictions.